"""
====================================================================================
Title: Deep Learning Models for Ozone Forecasting (MLP, GRU, HMSC, HMWF, HMAM)
Author: Naeem Ullah

====================================================================================

Description:
------------
This file contains the implementation of all deep learning models used for ozone
forecasting in the study. The models implemented include:

    • MLP  – Multilayer Perceptron  
    • GRU  – Gated Recurrent Unit  
    • HMSC – Hybrid model using simple concatenation 
    • HMWF – Hybrid model using weighted fusion
    • HMAM – Hybrid model using attention mechanism 

These architectures were designed to evaluate the effect of hybrid feature fusion
and temporal modeling on ozone concentration prediction.

Context of Use:
---------------
The code supports the experimental setup described in the paper’s *Performance Comparison*
section. It uses the **optimal feature combination (BM + LF + RWS)** identified for all
stations (Aljarafe, Asomadilla, Bermejales, Ronda del Valle, Torneo).

Since the same model architectures, feature engineering steps, and optimization methods
apply to all stations, **this single file can be reused for every dataset** — only the
dataset path needs to be updated in the preprocessing or data loading section.

Key Details:
-------------
- Input: Preprocessed dataset with BM + LF + RWS features.  
- Splitting: Chronological 80/20 split (training/testing).  
- Optimization: Bayesian hyperparameter tuning for each model.  
- Output: Model predictions, evaluation metrics (RMSE, MAE, MSE), and training logs.  

Usage Instructions:
-------------------
1. Update the dataset path (e.g., `DATA_PATH = "./data/Aljarafe.csv"`) in the data loading section.
2. Run the script to train and evaluate all implemented models.
3. The same workflow can be repeated for other stations by changing only the dataset path.

Example:
---------
# Example usage for Torneo dataset
DATA_PATH = "./data/Torneo.csv"
run_experiment(DATA_PATH)

====================================================================================
"""


MLP model Code

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ALJARAFE-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # Create lag features with a window size of 24
    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # Add rolling statistics (mean, std, skewness) with a window size of 24
    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\aljarafe_15-24 results\contaminacion_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()

GRU model Code

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GRU
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ALJARAFE-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # Create lag features with a window size of 24
    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # Add rolling statistics (mean, std, skewness) with a window size of 24
    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test, scaler, X

# Build the GRU-only model
def build_gru_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    input_layer = Input(shape=(input_dim, 1))  # Input for GRU requires 3D shape
    gru = GRU(num_units, activation=activation, return_sequences=False)(input_layer)
    gru = Dropout(dropout_rate)(gru)
    output = Dense(1)(gru)  # Output layer for regression
    
    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_gru_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(
        np.expand_dims(X_train, -1), 
        y_train, 
        epochs=epochs, 
        batch_size=batch_size, 
        validation_split=0.2, 
        verbose=0
    )
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(np.expand_dims(X_test, -1), verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\OneDrive_3_12-18-2024\contaminacion_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler, X = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final GRU-only model with the best parameters
        final_model = build_gru_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(
            np.expand_dims(X_train, -1), 
            y_train,   
            epochs=int(best_params['epochs']), 
            batch_size=int(best_params['batch_size']), 
            validation_split=0.2, 
            verbose=1
        )
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


Hybrid model with rolling averages, feature lagging, and Bayesian Optimization to find best hyperparameters 

Note: All these below codes are correct codes but as we wanted to get optimal performance so we experimented several approaches for hybridization of this model

Final correct code of concatenations: first attempt on hybridization, Simple concatenation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GRU, concatenate
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import math
from bayes_opt import BayesianOptimization

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)
    
    target_col = 'ALJARAFE-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)
    return df, target_col

# Split data
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test, scaler

# Hybrid model
def build_hybrid_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    input_layer = Input(shape=(input_dim,))
    mlp = Dense(num_units, activation=activation)(input_layer)
    mlp = Dropout(dropout_rate)(mlp)
    mlp = Dense(num_units // 2, activation=activation)(mlp)

    reshaped_input = Input(shape=(input_dim, 1))
    gru = GRU(num_units, activation=activation, return_sequences=False)(reshaped_input)

    combined = concatenate([mlp, gru])
    output = Dense(1)(combined)
    
    model = Model(inputs=[input_layer, reshaped_input], outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]
    model = build_hybrid_model(X_train.shape[1], int(num_units), dropout_rate, activation)
    history = model.fit(
        [X_train, np.expand_dims(X_train, -1)], y_train,
        epochs=int(epochs), batch_size=int(batch_size), validation_split=0.2, verbose=0
    )
    val_mae = np.min(history.history['val_mae'])
    return -val_mae

# Bayesian optimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),
        'dropout_rate': (0.1, 0.5),
        'activation': (0, 2),
        'epochs': (50, 100),
        'batch_size': (16, 64)
    }
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds, random_state=42
    )
    optimizer.maximize(init_points=5, n_iter=15)
    return optimizer.max['params']

# Evaluate model
def evaluate_model(model, X_test, y_test):
    predictions = model.predict([X_test, np.expand_dims(X_test, -1)])
    rmse = math.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    print(f"RMSE: {rmse}")
    print(f"MAE: {mae}")

# Training
def main():
    global X_train, X_test, y_train, y_test
    filepath = r"E:\Abroad period research\Time series forecasting\OneDrive_3_12-18-2024\contaminacion_2015_2023.ods"
    df, target_col = load_and_preprocess_data(filepath)
    X_train, X_test, y_train, y_test, _ = split_data(df, target_col)
    
    best_params = optimize_hyperparameters(X_train, y_train)
    print("Best Hyperparameters:", best_params)
    
    final_model = build_hybrid_model(
        X_train.shape[1], int(best_params['num_units']), best_params['dropout_rate'], ['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
    )
    final_model.fit([X_train, np.expand_dims(X_train, -1)], y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
    
    evaluate_model(final_model, X_test, y_test)
    
    # Save the model
    final_model.save("concatenated_model.h5")
    print("Model saved as concatenated_model.h5")

if __name__ == "__main__":
    main()


Second attempt of feature fusion: Hybrid model with Weighted Fusion

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GRU, concatenate, Multiply, Add, Reshape
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from bayes_opt import BayesianOptimization

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Parse datetime column
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Convert data to numeric and handle missing values
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)
    
    # Add lagged and rolling statistical features
    target_col = 'ALJARAFE-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)
    return df, target_col

# Split data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test, scaler

# Build the hybrid model with Weighted Fusion
def build_hybrid_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    # MLP branch
    input_layer = Input(shape=(input_dim,))
    mlp = Dense(num_units, activation=activation)(input_layer)
    mlp = Dropout(dropout_rate)(mlp)
    mlp = Dense(num_units // 2, activation=activation)(mlp)
    mlp = Dense(1)(mlp)  # Reduce dimensionality to match GRU output

    # GRU branch
    reshaped_input = Reshape((input_dim, 1))(input_layer)  # Reshape for GRU input
    gru = GRU(num_units, activation=activation, return_sequences=False)(reshaped_input)
    gru = Dense(1)(gru)  # Reduce dimensionality to match MLP output

    # Weighted fusion
    weight_mlp = Dense(1, activation='sigmoid', use_bias=False, name="mlp_weight")(mlp)
    weight_gru = Dense(1, activation='sigmoid', use_bias=False, name="gru_weight")(gru)

    weighted_mlp = Multiply()([mlp, weight_mlp])
    weighted_gru = Multiply()([gru, weight_gru])

    combined = Add()([weighted_mlp, weighted_gru])
    output = Dense(1)(combined)
    
    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Define the objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]
    model = build_hybrid_model(X_train.shape[1], int(num_units), dropout_rate, activation)
    history = model.fit(
        X_train, y_train,
        epochs=int(epochs), batch_size=int(batch_size), validation_split=0.2, verbose=0
    )
    val_mae = np.min(history.history['val_mae'])
    return -val_mae

# Optimize hyperparameters using Bayesian optimization
def optimize_hyperparameters():
    pbounds = {
        'num_units': (32, 128),
        'dropout_rate': (0.1, 0.5),
        'activation': (0, 2),  # Encoded activation function index
        'epochs': (50, 100),
        'batch_size': (16, 64)
    }
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: 
        objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds, random_state=42
    )
    optimizer.maximize(init_points=5, n_iter=15)
    return optimizer.max['params']

# Evaluate the model
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = math.sqrt(mse)
    print(f"Root Mean Squared Error (RMSE): {rmse}")
    print(f"Mean Absolute Error (MAE): {mae}")
    return rmse, mae

# Main function
def main():
    global X_train, X_test, y_train, y_test
    filepath = r"E:\Abroad period research\Time series forecasting\OneDrive_3_12-18-2024\contaminacion_2015_2023.ods"
    df, target_col = load_and_preprocess_data(filepath)
    X_train, X_test, y_train, y_test, _ = split_data(df, target_col)
    
    # Optimize hyperparameters
    best_params = optimize_hyperparameters()
    print("Best hyperparameters:", best_params)

    # Build and train the final model
    final_model = build_hybrid_model(
        X_train.shape[1], 
        int(best_params['num_units']), 
        best_params['dropout_rate'], 
        ['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
    )
    final_model.fit(
        X_train, y_train, 
        epochs=int(best_params['epochs']), 
        batch_size=int(best_params['batch_size']), 
        validation_split=0.2, 
        verbose=1
    )

    # Evaluate the final model
    rmse, mae = evaluate_model(final_model, X_test, y_test)

    # Save the model
    final_model.save("weighted_average_model.h5")
    print("Model saved as 'weighted_average_model.h5'")

# Run the script
if __name__ == "__main__":
    main()


Third hybridization approach: Hybridization using Attention Mechanism, We have Added an attention layer to assign importance to the features extracted by MLP and GRU before combining their outputs.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GRU, concatenate, Multiply, Activation, GlobalAveragePooling1D, Lambda
from tensorflow.keras import backend as K
from bayes_opt import BayesianOptimization

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    df = pd.read_excel(filepath, engine="odf")

    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)

    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)

    target_col = 'ALJARAFE-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()

    df.dropna(inplace=True)
    return df, target_col

# Split data
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test, scaler

# Attention Layer
def attention_layer(inputs):
    attention_scores = Dense(1, activation="tanh")(inputs)
    attention_weights = Activation("softmax")(attention_scores)
    context_vector = Multiply()([inputs, attention_weights])
    return context_vector

# Hybrid model with Attention Mechanism
def build_hybrid_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    input_layer = Input(shape=(input_dim,))
    
    # MLP branch
    mlp = Dense(num_units, activation=activation)(input_layer)
    mlp = Dropout(dropout_rate)(mlp)
    mlp = Dense(num_units // 2, activation=activation)(mlp)
    mlp_attention = attention_layer(mlp)

    # GRU branch
    reshaped_input = Lambda(lambda x: K.expand_dims(x, axis=-1))(input_layer)  # Corrected reshaping
    gru = GRU(num_units, activation=activation, return_sequences=True)(reshaped_input)
    gru_attention = attention_layer(gru)

    # Pooling GRU output to match MLP output shape
    gru_attention_pooled = GlobalAveragePooling1D()(gru_attention)

    # Combine both branches
    combined = concatenate([mlp_attention, gru_attention_pooled])
    output = Dense(1)(combined)

    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian Optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]
    model = build_hybrid_model(X_train.shape[1], int(num_units), dropout_rate, activation)
    history = model.fit(
        X_train, y_train,
        epochs=int(epochs), batch_size=int(batch_size), validation_split=0.2, verbose=0
    )
    val_mae = np.min(history.history['val_mae'])
    return -val_mae

# Bayesian optimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),
        'dropout_rate': (0.1, 0.5),
        'activation': (0, 2),
        'epochs': (50, 100),
        'batch_size': (16, 64)
    }
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds, random_state=42
    )
    optimizer.maximize(init_points=5, n_iter=15)
    return optimizer.max['params']

# Evaluate the model
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mse)
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

# Main function
def main():
    global X_train, X_test, y_train, y_test
    filepath = r"E:\Abroad period research\Time series forecasting\OneDrive_3_12-18-2024\contaminacion_2015_2023.ods"
    df, target_col = load_and_preprocess_data(filepath)
    X_train, X_test, y_train, y_test, _ = split_data(df, target_col)
    
    # Optimize hyperparameters
    best_params = optimize_hyperparameters(X_train, y_train)
    print("\nBest Hyperparameters:")
    print(best_params)
    
    # Build the final model with best hyperparameters
    final_model = build_hybrid_model(
        X_train.shape[1], 
        int(best_params['num_units']), 
        best_params['dropout_rate'], 
        ['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
    )
    
    # Train the final model
    final_model.fit(
        X_train, 
        y_train, 
        epochs=int(best_params['epochs']), 
        batch_size=int(best_params['batch_size']), 
        validation_split=0.2, 
        verbose=1
    )
    
    # Save the model
    final_model.save('final_attention_layer_model.h5')
    print("Model saved as 'final_model.h5'")
    
    # Evaluate the final model
    evaluate_model(final_model, X_test, y_test)

if __name__ == "__main__":
    main()
